# Metrohm Autolab (NOVA) Experimental Electrochemistry Workflow

This interactive research notebook provides an end-to-end scientific analysis pipeline for experimental electrochemistry datasets acquired from **Metrohm Autolab** potentiostats using **NOVA** software.

### Analysis Modules Covered:
1. **Oxygen Evolution Reaction (OER) LSV**: Calibration of reference potentials ($E$ vs RHE), software uncompensated resistance ($iR$) compensation, and overpotential extraction ($\eta_{10}, \eta_{50}, \eta_{100}$).
2. **Multi-Cycle Cyclic Voltammetry (CV)**: Peak detection ($E_{pa}, E_{pc}, \Delta E_p$) and voltammetric charge integration ($Q = \int I dt$).
3. **Electrochemical Impedance Spectroscopy (FRA EIS)**: Complex impedance polar conversion, publication Nyquist plots, and Randles circuit fitting ($R_s - [R_{ct} \parallel \mathrm{CPE}]$).
4. **Potentiostatic Chronoamperometry (CA)**: Long-term durability analysis, current retention percentage, and degradation rate (\%/h).


In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt

from scripts.utils import apply_publication_style, create_publication_figure
from parsers.electrochemistry import load_electrochemistry_data
from modules.electrochemistry import (
    plot_lsv,
    extract_overpotential_at_current,
    plot_cv,
    extract_redox_peaks,
    calculate_voltammetric_charge,
    plot_nyquist,
    fit_equivalent_circuit,
    calculate_impedance_polar,
    plot_chronoamperometry,
    calculate_ca_stability_metrics,
)

apply_publication_style()
data_dir = project_root / "data" / "examples" / "electrochemistry"
print("Environment successfully initialized with publication styles.")


## 1. Oxygen Evolution Reaction (OER) Linear Sweep Voltammetry

Load raw Metrohm Autolab ASCII export (`metrohm_oer_lsv.txt`), inspect automatically parsed metadata, and extract the overpotential delivering $10\,\mathrm{mA/cm^2}$ geometric current density ($\eta_{10}$).


In [ ]:
lsv_file = data_dir / "metrohm" / "metrohm_oer_lsv.txt"
data_oer = load_electrochemistry_data(lsv_file, instrument="metrohm", technique="LSV")

print(f"Instrument / Software: {data_oer.instrument_software}")
print(f"Working Electrode Area: {data_oer.electrode_area_cm2} cm^2")
print(f"Electrolyte pH: {data_oer.ph}")
print(f"Reference Electrode: {data_oer.potential_reference_raw}")
print(f"Uncompensated Resistance (Ru): {data_oer.ru_ohm} Ohm")

pot_rhe = data_oer.get_potential_vs_rhe()
j = data_oer.get_current_density(unit="mA/cm^2")

eta_10 = extract_overpotential_at_current(pot_rhe, j, target_j=10.0, reaction="OER")
eta_20 = extract_overpotential_at_current(pot_rhe, j, target_j=20.0, reaction="OER")
print(f"\n[Metrics] Overpotential eta_10: {eta_10:.1f} mV")
print(f"[Metrics] Overpotential eta_20: {eta_20:.1f} mV" if eta_20 is not None else "[Metrics] Overpotential eta_20: N/A")

ax, bm = plot_lsv(
    data_oer,
    reaction="OER",
    ir_compensate=False,
    target_currents=(10.0, 20.0),
    title="Metrohm Autolab - OER Polarization (1.0 M KOH)",
)
plt.show()


## 2. Multi-Cycle Cyclic Voltammetry (CV)

Analyze a 3-cycle CV scan (`metrohm_cv_multicycle.csv`), identify redox peak potentials ($E_{pa}, E_{pc}$), peak separation ($\Delta E_p$), and calculate voltammetric charge passed during cycling.


In [ ]:
cv_file = data_dir / "metrohm" / "metrohm_cv_multicycle.csv"
data_cv = load_electrochemistry_data(cv_file, instrument="metrohm", technique="CV")

cycle_3 = data_cv.get_cycle(3)
peaks = extract_redox_peaks(cycle_3.potential_V, cycle_3.current_A)
q_res = calculate_voltammetric_charge(cycle_3.time_s, cycle_3.current_A)

print(f"Anodic Peak Potential (E_pa): {peaks['anodic_peak_potential_V']:.3f} V | Current: {peaks['anodic_peak_current_A']:.4e} A")
print(f"Cathodic Peak Potential (E_pc): {peaks['cathodic_peak_potential_V']:.3f} V | Current: {peaks['cathodic_peak_current_A']:.4e} A")
print(f"Peak Potential Separation (Delta E_p): {peaks['delta_ep_mV']:.1f} mV")
print(f"Anodic Charge: {q_res['anodic_charge_C']:.4f} C | Cathodic Charge: {q_res['cathodic_charge_C']:.4f} C")

fig, ax = plot_cv(
    data_cv,
    cycles=[1, 2, 3],
    title="Metrohm Autolab - Multi-Cycle CV",
)
plt.show()


## 3. FRA Electrochemical Impedance Spectroscopy (EIS)

Fit high-frequency to low-frequency AC impedance data to a Randles equivalent circuit ($R_s - [R_{ct} \parallel \mathrm{CPE}]$) and display an orthonormal Nyquist representation.


In [ ]:
eis_file = data_dir / "metrohm" / "metrohm_fra_eis.txt"
data_eis = load_electrochemistry_data(eis_file, instrument="metrohm", technique="EIS")

f = data_eis.frequency_Hz
zr = data_eis.impedance_real_ohm
zi = data_eis.impedance_imag_ohm

fit_res = fit_equivalent_circuit(f, zr, zi)

print(f"Solution Resistance (R_s): {fit_res['rs_ohm']:.2f} Ohm")
print(f"Charge-Transfer Resistance (R_ct): {fit_res['rct_ohm']:.2f} Ohm")
print(f"Fit RMSE: {fit_res['rmse']:.3f} Ohm")

fig, ax = plot_nyquist(
    data_eis,
    fit_results=fit_res,
    title="Metrohm Autolab - FRA Nyquist Spectrum & Randles Fit",
)
plt.show()


## 4. Potentiostatic Chronoamperometry (CA) Stability

Evaluate catalyst durability under constant bias ($E = 1.60\,\mathrm{V}$ vs RHE). Compute current retention percentage and degradation rate (\%/h).


In [ ]:
ca_file = data_dir / "metrohm" / "metrohm_stability_ca.txt"
data_ca = load_electrochemistry_data(ca_file, instrument="metrohm", technique="CA")

t = data_ca.time_s
j = data_ca.get_current_density(unit="mA/cm^2")

metrics_ca = calculate_ca_stability_metrics(t, j)

print(f"Duration: {metrics_ca['duration_hours']:.2f} hours")
print(f"Current Retention: {metrics_ca['retention_percent']:.1f} %")
print(f"Degradation Rate: {metrics_ca['degradation_rate_pct_per_h']:.2f} %/h")

fig, ax = plot_chronoamperometry(
    data_ca,
    potential_label="E = 1.60 V vs RHE",
    show_retention=True,
    title="Metrohm Autolab - 1-Hour OER Durability",
)
plt.show()
